# 2024/2025 CFPB Data Ingestion and Validation

This notebook downloads separate local raw CFPB complaint datasets for the financial complaint auto-routing project using the reusable monthly-balanced + daily-stratified data-ingestion workflow.

Canonical local output paths:

- `data/raw/cfpb_complaints_2024_raw.csv`
- `data/raw/cfpb_complaints_2025_raw.csv`

Raw data remains local only and must not be committed to GitHub.

## 1. Year-Based Dataset Design

The 2024 dataset is the model-development dataset. It can later be split into train, validation, and internal test sets during modeling work.

The 2025 dataset is kept separate as a future holdout / out-of-time test dataset. It should not be mixed with 2024 and randomly split, and it should not be used during baseline model training or model selection.

This setup better simulates a business setting where models are developed on historical complaint data and tested on future complaints.

## 2. Why Monthly-Balanced + Daily-Stratified Sampling?

The original newest-first API sample covered only late December 2024, which created late-year recency bias. The next monthly-balanced approach improved year coverage, but each month was still pulled newest-first, creating within-month end-of-month bias.

This notebook uses monthly-balanced + daily-stratified sampling for each year. It keeps roughly equal monthly coverage, then allocates each month target across daily windows. This reduces both late-year recency bias and within-month end-of-month bias.

This notebook only performs data ingestion and validation. It does not perform Week 3 EDA, model training, charts, model scores, or confusion matrices.

## 3. Imports and Settings

In [7]:
from pathlib import Path
import sys

import pandas as pd
import requests

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.download_data import (
    REQUIRED_COLUMNS,
    build_daily_targets,
    build_monthly_targets,
    extract_records,
    fetch_cfpb_page,
    load_or_download_validate_year,
    raw_csv_relative_path,
)

YEARS = [2024, 2025]
TARGET_ROWS = 50_000
PAGE_SIZE = 1000
API_SMOKE_TEST_PAGE_SIZE = 50
MAX_PAGES_PER_DAY = 10
SLEEP_SECONDS = 0.05
FORCE_DOWNLOAD = False
RUN_API_SMOKE_TEST = False

print(f"FORCE_DOWNLOAD: {FORCE_DOWNLOAD}")
print(f"RUN_API_SMOKE_TEST: {RUN_API_SMOKE_TEST}")
print(f"API download page size: {PAGE_SIZE}")
print(f"API smoke test page size: {API_SMOKE_TEST_PAGE_SIZE}")
for year in YEARS:
    raw_path = PROJECT_ROOT / raw_csv_relative_path(year)
    print(f"{year} raw output path: {raw_csv_relative_path(year).as_posix()} (exists: {raw_path.exists()})")

FORCE_DOWNLOAD: False
RUN_API_SMOKE_TEST: False
API download page size: 1000
API smoke test page size: 50
2024 raw output path: data/raw/cfpb_complaints_2024_raw.csv (exists: True)
2025 raw output path: data/raw/cfpb_complaints_2025_raw.csv (exists: True)


## 4. Review Monthly and Daily Targets

For each year, January through November target 4,167 rows each. December is adjusted to 4,163 rows so the full target remains 50,000 rows.

Within each month, the monthly target is allocated as evenly as possible across that month's daily windows.

In [8]:
for year in YEARS:
    monthly_targets_df = pd.DataFrame(build_monthly_targets(year=year, total_rows=TARGET_ROWS))
    daily_targets_df = pd.DataFrame(build_daily_targets(year=year, total_rows=TARGET_ROWS))

    print()
    print(f"{year} monthly targets:")
    print(monthly_targets_df.to_string(index=False))
    print(f"Total monthly target rows: {monthly_targets_df['target_rows'].sum():,}")
    print(f"Daily windows: {len(daily_targets_df):,}")
    print(f"Total daily target rows: {daily_targets_df['target_rows'].sum():,}")
    print("Daily target range:", int(daily_targets_df['target_rows'].min()), "to", int(daily_targets_df['target_rows'].max()))


2024 monthly targets:
  month      start        end  target_rows
2024-01 2024-01-01 2024-01-31         4167
2024-02 2024-02-01 2024-02-29         4167
2024-03 2024-03-01 2024-03-31         4167
2024-04 2024-04-01 2024-04-30         4167
2024-05 2024-05-01 2024-05-31         4167
2024-06 2024-06-01 2024-06-30         4167
2024-07 2024-07-01 2024-07-31         4167
2024-08 2024-08-01 2024-08-31         4167
2024-09 2024-09-01 2024-09-30         4167
2024-10 2024-10-01 2024-10-31         4167
2024-11 2024-11-01 2024-11-30         4167
2024-12 2024-12-01 2024-12-31         4163
Total monthly target rows: 50,000
Daily windows: 366
Total daily target rows: 50,000
Daily target range: 134 to 144

2025 monthly targets:
  month      start        end  target_rows
2025-01 2025-01-01 2025-01-31         4167
2025-02 2025-02-01 2025-02-28         4167
2025-03 2025-03-01 2025-03-31         4167
2025-04 2025-04-01 2025-04-30         4167
2025-05 2025-05-01 2025-05-31         4167
2025-06 2025-06-01 20

## 5. Optional API Smoke Test

This optional smoke test checks that the CFPB API returns records with the required fields for the first daily window of each year. It is disabled by default so Run All can validate existing local ignored CSV files without requiring an API request. It prints schema-level information only and does not print complaint narrative examples.

In [9]:
if not RUN_API_SMOKE_TEST:
    print("API smoke test skipped. Set RUN_API_SMOKE_TEST = True to test the CFPB API.")
else:
    with requests.Session() as session:
        for year in YEARS:
            first_day = build_daily_targets(year=year, total_rows=TARGET_ROWS)[0]
            raw_path = PROJECT_ROOT / raw_csv_relative_path(year)
            fresh_download_required = FORCE_DOWNLOAD or not raw_path.exists()

            try:
                test_json = fetch_cfpb_page(
                    date_start=first_day["start"],
                    date_end=first_day["end"],
                    page_size=API_SMOKE_TEST_PAGE_SIZE,
                    session=session,
                )
                test_records = extract_records(test_json)
            except requests.exceptions.RequestException as exc:
                print(f"Warning: {year} API smoke test failed: {exc}")
                if fresh_download_required:
                    raise
                print(f"Continuing because existing local CSV is available for {year}.")
                continue

            if not test_records:
                message = f"No records returned for {year}. Check the API connection and query parameters."
                if fresh_download_required:
                    raise RuntimeError(message)
                print(f"Warning: {message}")
                print(f"Continuing because existing local CSV is available for {year}.")
                continue

            first_record = test_records[0]
            missing_columns = [column for column in REQUIRED_COLUMNS if column not in first_record]
            total_available = test_json.get("hits", {}).get("total", {}).get("value")

            print()
            print(f"{year} API test window: {first_day['start']}")
            print(f"Total matching API records in test window: {total_available:,}")
            print(f"Returned test records: {len(test_records)}")
            print("Required columns present:", missing_columns == [])
            print("Available API columns:")
            print(sorted(first_record.keys()))

            if missing_columns:
                raise ValueError(f"Missing required columns in {year} API response: {missing_columns}")

API smoke test skipped. Set RUN_API_SMOKE_TEST = True to test the CFPB API.


## 6. Load or Download 2024 and 2025 Raw Samples

This cell validates existing local raw CSVs when available. If a CSV is missing, or if `FORCE_DOWNLOAD = True`, it downloads that year separately using the reusable monthly-balanced + daily-stratified helper workflow. Each year is sampled independently, saved independently, and validated independently.

The saved CSV files remain ignored by Git through `.gitignore`.

In [10]:
download_results = {}

with requests.Session() as session:
    for year in YEARS:
        print()
        result = load_or_download_validate_year(
            year=year,
            project_root=PROJECT_ROOT,
            force_download=FORCE_DOWNLOAD,
            total_rows=TARGET_ROWS,
            page_size=PAGE_SIZE,
            max_pages_per_day=MAX_PAGES_PER_DAY,
            sleep_seconds=SLEEP_SECONDS,
            session=session,
            verbose=True,
        )
        download_results[year] = result

        monthly_log_df = pd.DataFrame(result["monthly_log"])
        daily_log_df = pd.DataFrame(result["daily_log"])

        print(f"Source: {result['source']}")
        if result["loaded_existing"]:
            print("Fresh API download skipped because local CSV exists and FORCE_DOWNLOAD is False.")
            print("Download shortfall/backfill logs are not available from an existing raw CSV.")
        else:
            print()
            print(f"{year} monthly download log:")
            print(monthly_log_df.to_string(index=False))
            print()
            print(f"{year} daily shortfall summary:")
            print("Daily windows with shortfall before backfill:", int((daily_log_df["shortfall_before_backfill"] > 0).sum()))
            print("Total daily shortfall before backfill:", int(daily_log_df["shortfall_before_backfill"].sum()))
            print("Rows backfilled within month:", int(daily_log_df["backfilled_rows"].sum()))
            print("Monthly shortfall after backfill:", int(monthly_log_df["month_shortfall_after_backfill"].sum()))

        print(f"Saved raw data path: {result['output_relative_path'].as_posix()}")
        print(f"Rows loaded: {len(result['dataframe']):,}")
        print(f"Columns loaded: {len(result['dataframe'].columns):,}")


Using existing local raw CSV for 2024: data/raw/cfpb_complaints_2024_raw.csv
Source: existing_csv
Fresh API download skipped because local CSV exists and FORCE_DOWNLOAD is False.
Download shortfall/backfill logs are not available from an existing raw CSV.
Saved raw data path: data/raw/cfpb_complaints_2024_raw.csv
Rows loaded: 50,000
Columns loaded: 17

Using existing local raw CSV for 2025: data/raw/cfpb_complaints_2025_raw.csv
Source: existing_csv
Fresh API download skipped because local CSV exists and FORCE_DOWNLOAD is False.
Download shortfall/backfill logs are not available from an existing raw CSV.
Saved raw data path: data/raw/cfpb_complaints_2025_raw.csv
Rows loaded: 50,000
Columns loaded: 17


## 7. Validate Saved Raw CSV Files

This validation confirms each raw local file is ready for later workflows. It reports aggregate quality checks only and does not print complaint text.

In [11]:
for year in YEARS:
    result = download_results[year]
    validation = result["validation"]
    rows_per_day = pd.Series(validation["rows_per_day"])
    monthly_log_df = pd.DataFrame(result["monthly_log"])
    daily_log_df = pd.DataFrame(result["daily_log"])

    print()
    print(f"{year} validation")
    print(f"Final row count: {validation['row_count']:,}")
    print(f"Column count: {validation['column_count']:,}")
    print(f"Actual date range: {validation['date_min']} to {validation['date_max']}")
    print("Rows per month:")
    for month, count in validation["rows_per_month"].items():
        print(f"  {month}: {count:,}")
    print(f"Unique dates covered: {validation['unique_dates_covered']:,} of {validation['expected_dates_in_year']:,}")
    print(f"Missing calendar dates: {len(validation['missing_dates']):,}")
    print(f"Rows per covered day range: {int(rows_per_day.min()):,} to {int(rows_per_day.max()):,}")
    print(f"Number of product classes: {validation['product_classes']:,}")
    print(f"Missing/empty complaint narratives: {validation['missing_empty_narratives']:,}")
    print(f"Missing/empty product labels: {validation['missing_empty_products']:,}")
    print(f"Duplicate complaint_id values: {validation['duplicate_complaint_ids']:,}")
    print(f"Rows outside {year}: {validation['rows_outside_year']:,}")
    print(f"All records within {year}: {validation['all_records_within_year']}")
    if daily_log_df.empty or monthly_log_df.empty:
        print("Download shortfall/backfill logs: not available from an existing raw CSV.")
    else:
        print("Daily windows with shortfall before backfill:", int((daily_log_df["shortfall_before_backfill"] > 0).sum()))
        print("Total daily shortfall before backfill:", int(daily_log_df["shortfall_before_backfill"].sum()))
        print("Rows backfilled within month:", int(daily_log_df["backfilled_rows"].sum()))
        print("Monthly shortfall after backfill:", int(monthly_log_df["month_shortfall_after_backfill"].sum()))

    assert validation["row_count"] > 0
    assert validation["row_count"] <= TARGET_ROWS
    assert validation["missing_empty_narratives"] == 0
    assert validation["missing_empty_products"] == 0
    assert validation["duplicate_complaint_ids"] == 0
    assert validation["all_records_within_year"]

print("Saved CSV validation passed for all years.")


2024 validation
Final row count: 50,000
Column count: 17
Actual date range: 2024-01-01 to 2024-12-31
Rows per month:
  2024-01: 4,167
  2024-02: 4,167
  2024-03: 4,167
  2024-04: 4,167
  2024-05: 4,167
  2024-06: 4,167
  2024-07: 4,167
  2024-08: 4,167
  2024-09: 4,167
  2024-10: 4,167
  2024-11: 4,167
  2024-12: 4,163
Unique dates covered: 366 of 366
Missing calendar dates: 0
Rows per covered day range: 134 to 144
Number of product classes: 11
Missing/empty complaint narratives: 0
Missing/empty product labels: 0
Duplicate complaint_id values: 0
Rows outside 2024: 0
All records within 2024: True
Download shortfall/backfill logs: not available from an existing raw CSV.

2025 validation
Final row count: 50,000
Column count: 17
Actual date range: 2025-01-01 to 2025-12-31
Rows per month:
  2025-01: 4,167
  2025-02: 4,167
  2025-03: 4,167
  2025-04: 4,167
  2025-05: 4,167
  2025-06: 4,167
  2025-07: 4,167
  2025-08: 4,167
  2025-09: 4,167
  2025-10: 4,167
  2025-11: 4,167
  2025-12: 4,163


## 8. Download Summary

After this notebook runs successfully, the local raw CSV files contain separate monthly-balanced + daily-stratified CFPB complaint samples for 2024 and 2025.

Use 2024 for model development. Keep 2025 separate for future holdout / out-of-time testing.

No EDA findings, model scores, charts, or confusion matrices are created in this notebook.